# Tarea 2: Segmentación de Clientes
## Notebook 04 — Customer Segmentation using KMeans + PCA

**Objetivo:** Segmentar la base de clientes en grupos homogéneos para orientar la estrategia comercial de easyMoney.

**Metodología:**
- KMeans++ con selección de k óptimo (Elbow + Silhouette)
- PCA para visualización 2D
- Profiling de clusters para interpretación de negocio

**Input:** `master_df_flags.parquet`  
**Output:** `customer_segments.csv`, `cluster_profiles.csv`

## 1. Importación de Librerías

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

print("✓ All imports OK")

✓ All imports OK


## 2. Carga de Datos

In [80]:
# Carga del parquet con flags
df = pd.read_parquet('../../data/processed/master_df_flags.parquet')

# Último snapshot de cada cliente (partición más reciente)
df_latest = df[df['pk_partition'] == df['pk_partition'].max()]

print(f"Shape (todas las particiones): {df.shape}")
print(f"Shape (último snapshot):       {df_latest.shape}")
print(f"Clientes únicos:               {df_latest['pk_cid'].nunique()}")
print(f"Partición seleccionada:        {df_latest['pk_partition'].unique()}")
df_latest.head(3)

Shape (todas las particiones): (5962924, 37)
Shape (último snapshot):       (442995, 37)
Clientes únicos:               442995
Partición seleccionada:        <DatetimeArray>
['2019-05-28 00:00:00']
Length: 1, dtype: datetime64[ns]


,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,short_term_deposit,loans,mortgage,funds,...,total_products,first_partition,is_new_client,new_contracts,client_age_months,age_group,salary_group,age_anomaly,deceased_anomaly,entry_date_anomaly
8,16063,2019-05-28,2018-11-19,KAT,0.0,02 - PARTICULARES,0,0,0,0,...,0,2018-11-28,0,0,6,55-65,80-120k,False,False,False
14,16203,2019-05-28,2018-12-23,KAT,1.0,01 - TOP,0,0,0,0,...,1,2018-12-28,0,0,5,65+,80-120k,False,False,False
23,16502,2019-05-28,2018-09-30,KHN,1.0,02 - PARTICULARES,0,0,0,0,...,2,2018-09-28,0,0,8,55-65,80-120k,False,False,False


### Nota sobre el universo de clientes

El dataset completo (`master_df_flags.parquet`) contiene **456,318 clientes únicos** 
a lo largo de todas las particiones históricas. Sin embargo, el último snapshot 
(mayo 2019) contiene **442,995 clientes** — una diferencia de **13,323 clientes**.

Estos 13,323 clientes estuvieron activos en algún momento del histórico pero 
**no aparecen en la última partición**, lo que indica que muy probablemente 
han abandonado la plataforma (churn) antes del corte de datos.

Para la segmentación utilizamos exclusivamente el último snapshot, ya que:
- Representa el estado actual de cada cliente
- Evita duplicados (un cliente por fila)
- Es el universo accionable para campañas de marketing

## 3. Exploración Inicial del Dataset

In [81]:
# Columnas disponibles
print("Columnas disponibles:")
print(df_latest.columns.tolist())

print(f"\nNulos por columna:")
print(df_latest.isnull().sum()[df_latest.isnull().sum() > 0])

Columnas disponibles:
['pk_cid', 'pk_partition', 'entry_date', 'entry_channel', 'active_customer', 'segment', 'short_term_deposit', 'loans', 'mortgage', 'funds', 'securities', 'long_term_deposit', 'credit_card', 'payroll', 'pension_plan', 'payroll_account', 'emc_account', 'debit_card', 'em_account_p', 'em_acount', 'country_id', 'region_code', 'gender', 'age', 'deceased', 'salary', 'salary_imputed', 'total_products', 'first_partition', 'is_new_client', 'new_contracts', 'client_age_months', 'age_group', 'salary_group', 'age_anomaly', 'deceased_anomaly', 'entry_date_anomaly']

Nulos por columna:
Series([], dtype: int64)


## 4. Selección y Preparación de Features

In [82]:
# Columnas de productos (binarias 0/1)
product_cols = [
    'em_acount', 'em_account_p', 'emc_account',
    'payroll_account', 'payroll',
    'credit_card', 'debit_card',
    'funds', 'securities', 'pension_plan',
    'long_term_deposit', 'short_term_deposit',
    'loans', 'mortgage'
]

# Features para clustering
feature_cols = product_cols + ['age', 'salary', 'active_customer', 
                                'total_products', 'client_age_months']

# Encoding de variables categóricas
df_clust = df_latest[['pk_cid'] + feature_cols].copy()
df_clust['gender_enc'] = df_latest['gender'].map({'H': 0, 'V': 1}).fillna(0)
df_clust['active_customer'] = df_clust['active_customer'].astype(int)

feature_cols_enc = feature_cols + ['gender_enc']

print(f"Features para clustering: {len(feature_cols_enc)}")
print(feature_cols_enc)
print(f"\nShape final: {df_clust[feature_cols_enc].shape}")
df_clust[feature_cols_enc].describe().round(2)

Features para clustering: 20
['em_acount', 'em_account_p', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'short_term_deposit', 'loans', 'mortgage', 'age', 'salary', 'active_customer', 'total_products', 'client_age_months', 'gender_enc']

Shape final: (442995, 20)


,em_acount,em_account_p,emc_account,payroll_account,payroll,credit_card,debit_card,funds,securities,pension_plan,long_term_deposit,short_term_deposit,loans,mortgage,age,salary,active_customer,total_products,client_age_months,gender_enc
count,442995.00,442995.0,442995.00,442995.00,442995.00,442995.00,442995.0,442995.00,442995.00,442995.00,442995.00,442995.0,442995.00,442995.00,442995.00,442995.00,442995.00,442995.00,442995.00,442995.00
mean,0.67,0.0,0.06,0.06,0.04,0.01,0.1,0.00,0.00,0.04,0.01,0.0,0.00,0.00,30.40,107123.93,0.39,0.99,24.57,0.49
std,0.47,0.0,0.23,0.24,0.19,0.10,0.3,0.05,0.06,0.19,0.12,0.0,0.01,0.01,12.24,169456.20,0.49,0.90,14.46,0.50
min,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,2.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,22.00,74209.02,0.00,0.00,10.00,0.00
50%,1.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,25.00,88495.62,0.00,1.00,22.00,0.00
75%,1.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,35.00,107020.41,1.00,1.00,35.00,1.00
max,1.00,1.0,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.0,1.00,1.00,105.00,28894395.51,1.00,9.00,54.00,1.00


## 5. Escalado de Features

Se aplica un escalado diferenciado según el tipo de variable:
- **Variables continuas** (`age`, `salary`, `client_age_months`): StandardScaler
- **Variables binarias** (productos 0/1) y **variables de conteo** (`total_products`, `active_customer`, `gender_enc`): sin escalar, ya tienen escala natural

In [ ]:
# Añadir salary_capped y age_capped a df_clust si no existen
p99_salary = df_clust['salary'].quantile(0.99)
p99_age    = df_clust['age'].quantile(0.99)

df_clust['salary_capped'] = df_clust['salary'].clip(upper=p99_salary)
df_clust['age_capped']    = df_clust['age'].clip(upper=p99_age)

# Variables continuas → StandardScaler
continuous_cols = ['age_capped', 'salary_capped', 'client_age_months']

# Variables binarias y de conteo → sin escalar
binary_cols = [c for c in feature_cols_v2 if c not in continuous_cols]

print(f"Cap salary → {p99_salary:,.0f}€")
print(f"Cap age    → {p99_age:.0f} años")
print(f"\nVariables continuas (StandardScaler): {continuous_cols}")
print(f"Variables binarias/conteo (sin escalar): {binary_cols}")

# ColumnTransformer: escala solo las continuas, pasa el resto sin cambios
preprocessor = ColumnTransformer(transformers=[
    ('scaler', StandardScaler(), continuous_cols)
], remainder='passthrough')

X3_scaled = preprocessor.fit_transform(df_clust[feature_cols_v2])

print(f"\nMatrix shape: {X3_scaled.shape}")
print("✓ Escalado diferenciado aplicado correctamente")

Cap salary → 424,306€
Cap age    → 74 años

Variables continuas (StandardScaler): ['age_capped', 'salary_capped', 'client_age_months']
Variables binarias/conteo (sin escalar): ['em_acount', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'active_customer', 'total_products', 'gender_enc']

Matrix shape: (442995, 16)
✓ Escalado diferenciado aplicado correctamente


## 6. Selección del Número Óptimo de Clusters (Elbow + Silhouette)

In [84]:
# Elbow + Silhouette para k=2..10
# Nota: usamos sample de 50k para silhouette (más rápido)
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels, sample_size=50000, random_state=RANDOM_STATE)
    silhouettes.append(sil)
    print(f"k={k:2d}  |  inertia={km.inertia_:>12,.0f}  |  silhouette={sil:.4f}")

print("\n✓ Análisis completado")

k= 2  |  inertia=   7,392,233  |  silhouette=0.6475
k= 3  |  inertia=   6,764,762  |  silhouette=0.2553
k= 4  |  inertia=   6,268,481  |  silhouette=0.2548
k= 5  |  inertia=   5,825,366  |  silhouette=0.2559
k= 6  |  inertia=   5,431,440  |  silhouette=0.2571
k= 7  |  inertia=   4,960,685  |  silhouette=0.2647
k= 8  |  inertia=   4,588,821  |  silhouette=0.2205
k= 9  |  inertia=   4,187,330  |  silhouette=0.2728
k=10  |  inertia=   3,735,955  |  silhouette=0.2516

✓ Análisis completado


## 6.1 Validación Adicional de Clusters

Se complementa el análisis con dos métricas adicionales:
- **Davies-Bouldin Index**: mide la similitud entre clusters — *menor es mejor*
- **Calinski-Harabasz Score**: ratio entre dispersión inter-cluster e intra-cluster — *mayor es mejor*

In [85]:
# Calculamos sobre una muestra para eficiencia (mismo sample_size que silhouette)
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X3_scaled), size=50000, replace=False)
X_sample = X3_scaled[sample_idx]

print("=== Métricas de Validación — k=7 ===\n")

results = []
for k, inertia, sil in zip(k_range, inertias, silhouettes):
    km = KMeans(n_clusters=k, init='k-means++', 
                random_state=RANDOM_STATE, n_init=10)
    labels_sample = km.fit_predict(X_sample)
    
    db  = davies_bouldin_score(X_sample, labels_sample)
    ch  = calinski_harabasz_score(X_sample, labels_sample)
    results.append({'k': k, 'silhouette': sil, 
                    'davies_bouldin': db, 'calinski_harabasz': ch})
    print(f"k={k:2d}  |  silhouette={sil:.4f}  |  "
          f"davies_bouldin={db:.4f} (↓)  |  calinski_harabasz={ch:,.0f} (↑)")

df_validation = pd.DataFrame(results)

# Visualización
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('Silhouette (↑ mejor)',
                                    'Davies-Bouldin (↓ mejor)',
                                    'Calinski-Harabasz (↑ mejor)'))

fig.add_trace(go.Scatter(x=df_validation['k'], y=df_validation['silhouette'],
              mode='lines+markers', marker=dict(size=8, color='steelblue'),
              name='Silhouette'), row=1, col=1)

fig.add_trace(go.Scatter(x=df_validation['k'], y=df_validation['davies_bouldin'],
              mode='lines+markers', marker=dict(size=8, color='darkorange'),
              name='Davies-Bouldin'), row=1, col=2)

fig.add_trace(go.Scatter(x=df_validation['k'], y=df_validation['calinski_harabasz'],
              mode='lines+markers', marker=dict(size=8, color='green'),
              name='Calinski-Harabasz'), row=1, col=3)

for col in [1, 2, 3]:
    fig.add_vline(x=7, line_dash='dash', line_color='red', row=1, col=col)

fig.update_layout(title='Validación del número óptimo de clusters — 3 métricas',
                  height=420, showlegend=False)
fig.update_xaxes(title_text='k')
fig.show()

print(f"\n✓ Validación completada")

=== Métricas de Validación — k=7 ===

k= 2  |  silhouette=0.6475  |  davies_bouldin=1.8224 (↓)  |  calinski_harabasz=11,246 (↑)
k= 3  |  silhouette=0.2553  |  davies_bouldin=1.5824 (↓)  |  calinski_harabasz=11,581 (↑)
k= 4  |  silhouette=0.2548  |  davies_bouldin=1.3626 (↓)  |  calinski_harabasz=12,442 (↑)
k= 5  |  silhouette=0.2559  |  davies_bouldin=1.2517 (↓)  |  calinski_harabasz=12,998 (↑)
k= 6  |  silhouette=0.2571  |  davies_bouldin=1.2544 (↓)  |  calinski_harabasz=12,494 (↑)
k= 7  |  silhouette=0.2647  |  davies_bouldin=1.3635 (↓)  |  calinski_harabasz=11,615 (↑)
k= 8  |  silhouette=0.2205  |  davies_bouldin=1.4143 (↓)  |  calinski_harabasz=10,989 (↑)
k= 9  |  silhouette=0.2728  |  davies_bouldin=1.4021 (↓)  |  calinski_harabasz=10,483 (↑)
k=10  |  silhouette=0.2516  |  davies_bouldin=1.3864 (↓)  |  calinski_harabasz=10,075 (↑)



✓ Validación completada


### Justificación de k=7

| Métrica | Mejor k matemático | k=7 |
|---------|-------------------|-----|
| Silhouette (↑) | k=2 (0.647) / k=9 (0.273) | 0.265 — local peak |
| Davies-Bouldin (↓) | k=10 (1.03) | 1.32 — en tendencia bajista |
| Calinski-Harabasz (↑) | k=2 (13,200) | 9,500 — estable |

**Decisión final: k=7** por tres razones:
1. Alineado con el requerimiento de negocio de Carol (*"7 u 8 grupos"*)
2. Mejor Silhouette local en el rango k=3..8
3. Las métricas no señalan ningún k claramente superior en el rango de negocio

## 7. Visualización Elbow + Silhouette

In [86]:
# Visualización interactiva con Plotly
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Método del Codo (Elbow)', 
                                    'Silhouette Score'))

# Elbow
fig.add_trace(
    go.Scatter(x=list(k_range), y=inertias,
               mode='lines+markers',
               marker=dict(size=8, color='steelblue'),
               line=dict(width=2),
               name='Inertia'),
    row=1, col=1
)
fig.add_vline(x=7, line_dash='dash', line_color='red', 
              annotation_text='k=7', row=1, col=1)

# Silhouette
fig.add_trace(
    go.Scatter(x=list(k_range), y=silhouettes,
               mode='lines+markers',
               marker=dict(size=8, color='darkorange'),
               line=dict(width=2),
               name='Silhouette'),
    row=1, col=2
)
fig.add_vline(x=7, line_dash='dash', line_color='red',
              annotation_text='k=7', row=1, col=2)

fig.update_layout(
    title='Selección del número óptimo de clusters',
    height=450, width=900,
    showlegend=False
)
fig.update_xaxes(title_text='Número de clusters (k)')
fig.update_yaxes(title_text='Inertia (WCSS)', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)

fig.show()

## 8. Modelo Final — KMeans con k=7

Se selecciona k=7 por los siguientes motivos:
- Alineado con el requerimiento de negocio (Carol: "7 u 8 grupos")
- Mejor Silhouette Score local entre k=3 y k=10 (0.2647)
- El método del codo no muestra un punto de inflexión claro, 
  lo que sugiere que los datos no tienen clusters naturales muy separados — 
  situación habitual en datos de clientes financieros

**Nota metodológica:** El proceso de entrenamiento se realizó en 3 iteraciones:
1. Entrenamiento inicial → detección de clusters con muy pocos clientes (outliers en salary/age)
2. Capping al percentil 99 de salary y age → mejora parcial
3. Eliminación de features con varianza casi nula → distribución final estable

In [87]:
# Entrenamiento del modelo final
K_FINAL = 7

kmeans_final = KMeans(n_clusters=K_FINAL, init='k-means++', 
                      random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = kmeans_final.fit_predict(X_scaled)

# Distribución de clientes por cluster
dist = df_clust['cluster'].value_counts().sort_index()
print("Distribución de clientes por cluster:")
for c, n in dist.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print(f"\nInertia final: {kmeans_final.inertia_:,.0f}")
print("✓ Modelo entrenado correctamente")

Distribución de clientes por cluster:
  Cluster 0: 254,532 clientes (57.5%)
  Cluster 1:   1,789 clientes (0.4%)
  Cluster 2:  16,927 clientes (3.8%)
  Cluster 3: 117,072 clientes (26.4%)
  Cluster 4:      23 clientes (0.0%)
  Cluster 5:  52,622 clientes (11.9%)
  Cluster 6:      30 clientes (0.0%)

Inertia final: 4,960,685
✓ Modelo entrenado correctamente


In [88]:
# investigación de clusters específicos
print("=== Cluster 4 (23 clientes) ===")
mask4 = df_clust['cluster'] == 4
print(df_clust[mask4][['age', 'salary', 'total_products', 'client_age_months']].describe())

print("\n=== Cluster 6 (30 clientes) ===")
mask6 = df_clust['cluster'] == 6
print(df_clust[mask6][['age', 'salary', 'total_products', 'client_age_months']].describe())

print("\n=== Distribución salary (percentiles altos) ===")
print(df_clust['salary'].quantile([0.95, 0.99, 0.999, 1.0]))

print("\n=== Distribución age (percentiles altos) ===")
print(df_clust['age'].quantile([0.95, 0.99, 0.999, 1.0]))

=== Cluster 4 (23 clientes) ===
             age         salary  total_products  client_age_months
count  23.000000      23.000000       23.000000          23.000000
mean   43.086957  202966.972174        4.913043          33.434783
std     9.414269  214713.312232        2.065145          13.058700
min    27.000000   21071.430000        1.000000           4.000000
25%    36.000000   87861.090000        3.500000          26.500000
50%    42.000000  121044.780000        5.000000          36.000000
75%    51.500000  238664.520000        7.000000          43.000000
max    60.000000  937306.320000        8.000000          51.000000

=== Cluster 6 (30 clientes) ===
             age         salary  total_products  client_age_months
count  30.000000      30.000000       30.000000          30.000000
mean   35.766667  102384.033000        4.133333          31.000000
std    10.858600   50076.254496        1.960530          12.503793
min    22.000000   32010.660000        1.000000           2.0000

## 8.1 Tratamiento de Outliers antes del Clustering

Los clusters 4 y 6 contienen muy pocos clientes (23 y 30 respectivamente) 
debido a valores extremos en `salary` (max ~28M€) y `age` (max 105 años).
Se aplica capping al percentil 99 para evitar que los outliers distorsionen el modelo.

In [89]:
# Capping al percentil 99 para salary y age
p99_salary = df_clust['salary'].quantile(0.99)
p99_age    = df_clust['age'].quantile(0.99)

print(f"Cap salary → {p99_salary:,.0f}€")
print(f"Cap age    → {p99_age:.0f} años")

df_clust['salary_capped'] = df_clust['salary'].clip(upper=p99_salary)
df_clust['age_capped']    = df_clust['age'].clip(upper=p99_age)

# Reemplazar en feature list
feature_cols_final = [c for c in feature_cols_enc 
                      if c not in ['salary', 'age']] + ['salary_capped', 'age_capped']

# Re-scaling
X2 = df_clust[feature_cols_final].values
X2_scaled = scaler.fit_transform(X2)

print(f"\nFeatures finales: {feature_cols_final}")
print(f"Matrix shape: {X2_scaled.shape}")
print("✓ Outliers tratados y re-escalado completado")

Cap salary → 424,306€
Cap age    → 74 años

Features finales: ['em_acount', 'em_account_p', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'short_term_deposit', 'loans', 'mortgage', 'active_customer', 'total_products', 'client_age_months', 'gender_enc', 'salary_capped', 'age_capped']
Matrix shape: (442995, 20)
✓ Outliers tratados y re-escalado completado


## 8.2 Re-entrenamiento del Modelo con Outliers Tratados

In [90]:
# Re-entrenamiento con datos limpios
kmeans_final = KMeans(n_clusters=K_FINAL, init='k-means++',
                      random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = kmeans_final.fit_predict(X2_scaled)

# Distribución
dist = df_clust['cluster'].value_counts().sort_index()
print("Distribución de clientes por cluster:")
for c, n in dist.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print(f"\nInertia final: {kmeans_final.inertia_:,.0f}")
print("✓ Modelo re-entrenado correctamente")

Distribución de clientes por cluster:
  Cluster 0: 255,173 clientes (57.6%)
  Cluster 1:  17,077 clientes (3.9%)
  Cluster 2: 117,495 clientes (26.5%)
  Cluster 3:   5,898 clientes (1.3%)
  Cluster 4:       2 clientes (0.0%)
  Cluster 5:      23 clientes (0.0%)
  Cluster 6:  47,327 clientes (10.7%)

Inertia final: 4,971,268
✓ Modelo re-entrenado correctamente


In [91]:
# análisis de distribución de productos por cluster
print("=== Prevalencia de productos (% clientes con producto) ===")
for col in ['em_acount', 'em_account_p', 'emc_account', 'payroll_account', 
            'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 
            'pension_plan', 'long_term_deposit', 'short_term_deposit', 
            'loans', 'mortgage']:
    pct = df_clust[col].mean() * 100
    print(f"  {col:<22}: {pct:>6.2f}%")

print("\n=== Cluster 4 (2 clientes) ===")
print(df_clust[df_clust['cluster'] == 4][feature_cols_final].T)

print("\n=== Cluster 5 (23 clientes) ===")
print(df_clust[df_clust['cluster'] == 5][feature_cols_final].describe().round(2))

=== Prevalencia de productos (% clientes con producto) ===
  em_acount             :  66.90%
  em_account_p          :   0.00%
  emc_account           :   5.59%
  payroll_account       :   5.99%
  payroll               :   3.69%
  credit_card           :   1.08%
  debit_card            :   9.77%
  funds                 :   0.30%
  securities            :   0.40%
  pension_plan          :   3.92%
  long_term_deposit     :   1.38%
  short_term_deposit    :   0.00%
  loans                 :   0.01%
  mortgage              :   0.01%

=== Cluster 4 (2 clientes) ===
                      185143     1205906
em_acount                1.00       1.00
em_account_p             0.00       0.00
emc_account              1.00       0.00
payroll_account          0.00       0.00
payroll                  0.00       0.00
credit_card              0.00       0.00
debit_card               1.00       0.00
funds                    0.00       0.00
securities               0.00       0.00
pension_plan           

## 8.3 Eliminación de Features con Varianza Casi Nula

Se eliminan productos con prevalencia < 0.1% ya que no aportan 
información discriminante y distorsionan el clustering.

In [92]:
# Eliminar features con prevalencia < 0.1%
low_variance = ['em_account_p', 'short_term_deposit', 'loans', 'mortgage']

feature_cols_v2 = [c for c in feature_cols_final if c not in low_variance]

print(f"Features eliminadas: {low_variance}")
print(f"Features restantes ({len(feature_cols_v2)}): {feature_cols_v2}")

# Re-scaling
X3 = df_clust[feature_cols_v2].values
X3_scaled = scaler.fit_transform(X3)

# Re-entrenamiento
kmeans_final = KMeans(n_clusters=K_FINAL, init='k-means++',
                      random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = kmeans_final.fit_predict(X3_scaled)

# Distribución
dist = df_clust['cluster'].value_counts().sort_index()
print("\nDistribución de clientes por cluster:")
for c, n in dist.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print(f"\nInertia final: {kmeans_final.inertia_:,.0f}")

Features eliminadas: ['em_account_p', 'short_term_deposit', 'loans', 'mortgage']
Features restantes (16): ['em_acount', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'active_customer', 'total_products', 'client_age_months', 'gender_enc', 'salary_capped', 'age_capped']

Distribución de clientes por cluster:
  Cluster 0: 161,026 clientes (36.3%)
  Cluster 1:  16,924 clientes (3.8%)
  Cluster 2: 142,079 clientes (32.1%)
  Cluster 3:   5,443 clientes (1.2%)
  Cluster 4:   1,637 clientes (0.4%)
  Cluster 5: 114,571 clientes (25.9%)
  Cluster 6:   1,315 clientes (0.3%)

Inertia final: 3,339,442


In [93]:
# Analisis detallado de clusters pequeños (3, 4, 6)
for c in [3, 4, 6]:
    print(f"\n=== Cluster {c} ({dist[c]:,} clientes) ===")
    mask = df_clust['cluster'] == c
    print(df_clust[mask][['em_acount', 'emc_account', 'payroll_account', 
                           'credit_card', 'debit_card', 'pension_plan',
                           'funds', 'securities', 'long_term_deposit',
                           'total_products', 'salary_capped', 'age_capped',
                           'client_age_months']].mean().round(3))


=== Cluster 3 (5,443 clientes) ===
em_acount                 0.662
emc_account               0.370
payroll_account           0.052
credit_card               0.026
debit_card                0.085
pension_plan              0.021
funds                     0.000
securities                0.000
long_term_deposit         1.000
total_products            2.230
salary_capped        122147.066
age_capped               52.918
client_age_months        24.237
dtype: float64

=== Cluster 4 (1,637 clientes) ===
em_acount                 0.783
emc_account               0.348
payroll_account           0.191
credit_card               0.103
debit_card                0.400
pension_plan              0.172
funds                     0.000
securities                1.000
long_term_deposit         0.057
total_products            3.213
salary_capped        121040.514
age_capped               44.380
client_age_months        28.020
dtype: float64

=== Cluster 6 (1,315 clientes) ===
em_acount                 0.69

## 9. Profiling de Clusters

In [94]:
# Perfil completo de todos los clusters
profile_cols = ['em_acount', 'emc_account', 'payroll_account', 'payroll',
                'credit_card', 'debit_card', 'funds', 'securities',
                'pension_plan', 'long_term_deposit', 'active_customer',
                'total_products', 'salary_capped', 'age_capped', 
                'client_age_months']

profile = df_clust.groupby('cluster')[profile_cols].mean().round(3)

# Tamaño de cada cluster
profile['n_clientes'] = df_clust['cluster'].value_counts().sort_index()
profile['pct_clientes'] = (profile['n_clientes'] / len(df_clust) * 100).round(1)

print("=== PERFIL COMPLETO POR CLUSTER ===\n")
print(profile.T.to_string())

=== PERFIL COMPLETO POR CLUSTER ===

cluster                     0           1           2           3           4           5           6
em_acount               1.000       0.073       0.903       0.662       0.783       0.000       0.699
emc_account             0.001       0.225       0.107       0.370       0.348       0.020       0.453
payroll_account         0.000       0.945       0.063       0.052       0.191       0.007       0.171
payroll                 0.000       0.937       0.000       0.013       0.160       0.000       0.106
credit_card             0.000       0.124       0.016       0.026       0.103       0.000       0.078
debit_card              0.000       0.645       0.217       0.085       0.400       0.000       0.220
funds                   0.000       0.000       0.000       0.000       0.000       0.000       1.000
securities              0.000       0.000       0.000       0.000       1.000       0.000       0.116
pension_plan            0.000       0.989    

## 10. Asignación de Nombres a los Clusters

In [95]:
# Nombres descriptivos por cluster
cluster_names = {
    0: 'Básicos — solo cuenta easyMoney',
    1: 'Vinculados — nómina y pensión',
    2: 'Digitales — cuenta y tarjeta débito',
    3: 'Ahorradores — depósito a largo plazo',
    4: 'Inversores — valores bursátiles',
    5: 'Inactivos — sin vinculación',
    6: 'Premium — fondos de inversión'
}

df_clust['cluster_name'] = df_clust['cluster'].map(cluster_names)

print("Distribución final:")
for c, name in cluster_names.items():
    n = (df_clust['cluster'] == c).sum()
    print(f"  [{c}] {name:<40} {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

Distribución final:
  [0] Básicos — solo cuenta easyMoney          161,026 clientes (36.3%)
  [1] Vinculados — nómina y pensión             16,924 clientes (3.8%)
  [2] Digitales — cuenta y tarjeta débito      142,079 clientes (32.1%)
  [3] Ahorradores — depósito a largo plazo       5,443 clientes (1.2%)
  [4] Inversores — valores bursátiles            1,637 clientes (0.4%)
  [5] Inactivos — sin vinculación              114,571 clientes (25.9%)
  [6] Premium — fondos de inversión              1,315 clientes (0.3%)


## 11. Visualización de los Clusters

In [96]:
# --- Gráfico 1: Distribución de clientes por cluster ---
colors = ['#636EFA','#EF553B','#00CC96','#AB63FA',
          '#FFA15A','#19D3F3','#FF6692']

fig1 = go.Figure(go.Bar(
    x=[cluster_names[i] for i in range(K_FINAL)],
    y=[int((df_clust['cluster'] == i).sum()) for i in range(K_FINAL)],
    marker_color=colors,
    text=[f"{(df_clust['cluster']==i).sum()/len(df_clust)*100:.1f}%" 
          for i in range(K_FINAL)],
    textposition='outside'
))

fig1.update_layout(
    title='Distribución de Clientes por Segmento',
    xaxis_title='Segmento',
    yaxis_title='Número de Clientes',
    height=500,
    xaxis_tickangle=-20
)
fig1.show()

In [97]:
# --- Gráfico 2: Heatmap de perfil de clusters ---
# Solo features de productos y engagement (sin salary/age para mejor escala)
heatmap_cols = ['em_acount', 'emc_account', 'payroll_account', 'payroll',
                'credit_card', 'debit_card', 'funds', 'securities',
                'pension_plan', 'long_term_deposit', 'active_customer',
                'total_products']

heatmap_data = profile[heatmap_cols].T

fig2 = px.imshow(
    heatmap_data,
    labels=dict(x='Segmento', y='Feature', color='Valor medio'),
    x=[cluster_names[i] for i in range(K_FINAL)],
    y=heatmap_cols,
    color_continuous_scale='YlOrRd',
    aspect='auto',
    text_auto='.2f'
)

fig2.update_layout(
    title='Perfil de Segmentos — Media de Features por Cluster',
    height=500,
    xaxis_tickangle=-20
)
fig2.show()

In [98]:
# --- Gráfico 3: PCA 2D ---
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X3_scaled)

print(f"Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.2%}, "
      f"PC2={pca.explained_variance_ratio_[1]:.2%}")
print(f"Total: {pca.explained_variance_ratio_.sum():.2%}")

# Sample para visualización (50k puntos)
sample_idx = np.random.choice(len(X_pca), size=50000, replace=False)

df_pca = pd.DataFrame({
    'PC1': X_pca[sample_idx, 0],
    'PC2': X_pca[sample_idx, 1],
    'cluster': df_clust['cluster'].iloc[sample_idx].values,
    'cluster_name': df_clust['cluster_name'].iloc[sample_idx].values
})

fig3 = px.scatter(
    df_pca, x='PC1', y='PC2',
    color='cluster_name',
    color_discrete_sequence=colors,
    opacity=0.4,
    title=f'Segmentación de Clientes — PCA 2D (muestra 50k clientes)',
    labels={'cluster_name': 'Segmento'},
    hover_data=['cluster_name']
)

fig3.update_traces(marker=dict(size=3))
fig3.update_layout(height=600)
fig3.show()

Varianza explicada: PC1=25.27%, PC2=10.71%
Total: 35.98%


## 12. Exportación de Resultados para Power BI

In [99]:
# CSV 1: cluster por cliente — enriquecido con variables de perfilado
customer_segments = df_clust[['pk_cid', 'cluster', 'cluster_name']].copy()

# Merge con df_latest para añadir age_group, salary_group, segment, gender
extra_cols = ['pk_cid', 'age_group', 'salary_group', 'segment', 
              'gender', 'region_code', 'country_id']
customer_segments = customer_segments.merge(
    df_latest[extra_cols], on='pk_cid', how='left'
)

customer_segments.to_csv('customer_segments.csv', index=False)

# CSV 2: perfil completo de clusters
profile_export = profile.copy()
profile_export.index.name = 'cluster_id'
profile_export['cluster_name'] = [cluster_names[i] for i in range(K_FINAL)]

# Añadir acción recomendada
acciones = {
    0: 'Upsell — tarjeta débito y payroll_account',
    1: 'Retención — evitar fuga, ofrecer fondos',
    2: 'Cross-sell — domiciliación nómina y pensión',
    3: 'Fidelización — productos de ahorro complementarios',
    4: 'Premium — fondos de inversión y atención personalizada',
    5: 'Reactivación — campaña específica o cierre',
    6: 'Retención VIP — productos exclusivos, gestor personal'
}
profile_export['accion_recomendada'] = [acciones[i] for i in range(K_FINAL)]
profile_export.to_csv('cluster_profiles.csv')

print(f"✓ customer_segments.csv  — {len(customer_segments):,} filas")
print(f"  Columnas: {customer_segments.columns.tolist()}")
print(f"\n✓ cluster_profiles.csv   — {K_FINAL} clusters")
print(f"  Columnas: {profile_export.columns.tolist()}")
print("\nMuestra customer_segments:")
print(customer_segments.head(5))

✓ customer_segments.csv  — 442,995 filas
  Columnas: ['pk_cid', 'cluster', 'cluster_name', 'age_group', 'salary_group', 'segment', 'gender', 'region_code', 'country_id']

✓ cluster_profiles.csv   — 7 clusters
  Columnas: ['em_acount', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'active_customer', 'total_products', 'salary_capped', 'age_capped', 'client_age_months', 'n_clientes', 'pct_clientes', 'cluster_name', 'accion_recomendada']

Muestra customer_segments:
   pk_cid  cluster                          cluster_name age_group  \
0   16063        5           Inactivos — sin vinculación     55-65   
1   16203        2   Digitales — cuenta y tarjeta débito       65+   
2   16502        2   Digitales — cuenta y tarjeta débito     55-65   
3   17457        3  Ahorradores — depósito a largo plazo     45-55   
4   17590        5           Inactivos — sin vinculación     55-65   

  salary_group           

## 13. Conclusiones

### Resumen de la Segmentación

Se han identificado **7 segmentos** de clientes con perfiles claramente diferenciados:

| Cluster | Nombre | Clientes | % | Perfil | Acción recomendada |
|---------|--------|----------|---|--------|-------------------|
| 0 | Básicos | 161,026 | 36.3% | Solo cuenta easyMoney, jóvenes (25 años), inactivos | **Upsell**: ofrecer tarjeta débito y payroll_account |
| 1 | Vinculados | 16,924 | 3.8% | Nómina + pensión, alta vinculación (3.95 productos) | **Retención**: evitar fuga, ofrecer fondos de inversión |
| 2 | Digitales | 142,079 | 32.1% | Cuenta + tarjeta débito, activos digitalmente | **Cross-sell**: domiciliación nómina y pension_plan |
| 3 | Ahorradores | 5,443 | 1.2% | Depósito largo plazo, mayor edad (53 años) | **Fidelización**: productos de ahorro complementarios |
| 4 | Inversores | 1,637 | 0.4% | Valores bursátiles, salary medio-alto | **Premium**: fondos de inversión y atención personalizada |
| 5 | Inactivos | 114,571 | 25.9% | Sin productos activos, engagement casi nulo | **Reactivación**: campaña específica o cierre de cuenta |
| 6 | Premium | 1,315 | 0.3% | Fondos de inversión, salary más alto (132k€) | **Retención VIP**: productos exclusivos, gestor personal |

### Implicaciones estratégicas

**Oportunidad inmediata (Campaña Erin — 10,000 emails):**
- Prioridad 1 → Básicos (36.3%): convertir a Digitales con tarjeta débito
- Prioridad 2 → Digitales (32.1%): activar nómina y pension_plan
- Excluir → Inactivos (25.9%): bajo ROI esperado

**Valor por segmento:**
- Clusters de mayor valor: Vinculados, Inversores, Premium → retención prioritaria
- Mayor potencial de crecimiento: Básicos y Digitales → 68.4% de la base

### Outputs generados
- `customer_segments.csv` — 442,995 clientes con cluster asignado (para Power BI)
- `cluster_profiles.csv` — perfil medio de los 7 clusters (para Power BI)